# Día 8 · Unidad 7 (cierre) — pandas avanzado (II): `concat`, `merge`, `pivot_table` e import/export

**Objetivos de aprendizaje**
- Combinar DataFrames por filas con `pd.concat()`.
- Unir DataFrames por una clave común con `.merge()` (el equivalente de pandas a un `JOIN` de SQL o un `BUSCARV`).
- Construir tablas dinámicas con `pivot_table()`.
- Exportar resultados a CSV y Excel.
- **Pedir a la IA una transformación con `groupby`/`merge` y validarla contra un cálculo de control** — el primero de los dos bloques de IA del día.

**Duración estimada:** 100-120 minutos.

**Requisitos previos:** `01_teoria_groupby_avanzado.ipynb` de este mismo día.

In [1]:
import numpy as np
import pandas as pd

cartera = pd.read_csv("../data/raw/cartera_polizas.csv", parse_dates=["fecha_alta", "fecha_baja"])
siniestros = pd.read_csv("../data/raw/siniestros.csv", parse_dates=["fecha_siniestro"])
primas_mensuales = pd.read_csv("../data/raw/primas_mensuales.csv")

print(cartera.shape, siniestros.shape, primas_mensuales.shape)

(3000, 12) (1200, 8) (192, 5)


## 1. `pd.concat()`: combinar por filas (o columnas)

[`pd.concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) apila varios DataFrames o Series. Es el caso típico de "tengo los datos de dos periodos en dos archivos distintos y quiero trabajarlos como uno solo".

In [2]:
# Simulamos dos "cargas" separadas de la cartera, como si vinieran de dos exportaciones distintas
cartera_2023 = cartera[cartera["fecha_alta"].dt.year == 2023]
cartera_2024 = cartera[cartera["fecha_alta"].dt.year == 2024]

print(f"Altas en 2023: {len(cartera_2023)}, altas en 2024: {len(cartera_2024)}")

cartera_23_24 = pd.concat([cartera_2023, cartera_2024])
print(f"Concatenado: {len(cartera_23_24)} filas")

Altas en 2023: 739, altas en 2024: 730
Concatenado: 1469 filas


Fijaos en los índices: al concatenar, pandas **conserva** los índices originales de cada trozo, así que pueden quedar duplicados (el índice `0` de `cartera_2023` y el índice `0` de `cartera_2024`, si existiera, coexistirían).

In [3]:
print("¿Hay índices duplicados tras concat?", cartera_23_24.index.duplicated().any())

¿Hay índices duplicados tras concat? False


Dos formas de resolverlo, según lo que os interese conservar:

In [4]:
# Opción 1: reiniciar el índice (perdemos la referencia al índice original)
cartera_23_24_reset = pd.concat([cartera_2023, cartera_2024], ignore_index=True)
print("Índice tras ignore_index=True:", cartera_23_24_reset.index[:5].tolist(), "...")

# Opción 2: añadir un segundo nivel de índice que identifica de qué trozo viene cada fila
cartera_23_24_keys = pd.concat([cartera_2023, cartera_2024], keys=["2023", "2024"])
cartera_23_24_keys.head(2)

Índice tras ignore_index=True: [0, 1, 2, 3, 4] ...


poliza_id asegurado_id    nombre_asegurado  edad_tomador producto  \
2023 1  POL100001      AS20001  Vasco Godoy Rivero            54    HOGAR   
     2  POL100002      AS20002  Maite Tudela-Sainz            22    HOGAR   

       provincia canal_venta fecha_alta fecha_baja  prima_anual  franquicia  \
2023 1    Madrid      online 2023-06-25 2024-04-01       274.72         300   
     2  Valencia      agente 2023-07-02        NaT       207.32         150   

        anulada  
2023 1     True  
     2    False

> ⚠️ **Error típico**: concatenar y seguir usando el DataFrame resultante como si el índice fuera único (por ejemplo, con `.loc[0]` esperando una única fila) sin haberlo comprobado antes. Si hay índices duplicados, `.loc[0]` puede devolver **varias** filas en vez de una, y el código que sigue puede fallar de forma confusa más adelante — otro ejemplo del patrón "falla lejos de la causa real" que ya visteis en el Día 5.

## 2. `.merge()`: unir por una clave común

[`.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) es el equivalente de pandas a un `JOIN` de SQL — o, en términos de Excel, a un `BUSCARV` que trae **todas** las columnas coincidentes de golpe, en vez de una a una.

Vamos a unir `cartera` (una fila por póliza) con `siniestros` (una fila por siniestro, con `poliza_id` como clave foránea) — no todas las pólizas tienen siniestros.

In [5]:
cartera.head(2)[["poliza_id", "producto", "prima_anual"]]

,poliza_id,producto,prima_anual
0,POL100000,AUTO,455.69
1,POL100001,HOGAR,274.72


In [6]:
siniestros.head(2)[["siniestro_id", "poliza_id", "importe_pagado"]]

,siniestro_id,poliza_id,importe_pagado
0,SIN500001,POL100007,2096.22
1,SIN500002,POL100011,4521.37


### `how="left"`: conservar todas las pólizas

Un *left join* conserva **todas** las filas de `cartera` (la tabla "izquierda"), rellenando con `NaN` donde no hay siniestro asociado — el equivalente exacto de un `BUSCARV` que devuelve error/blanco cuando no encuentra coincidencia, pero sin perder la fila de la tabla principal.

In [7]:
cartera_con_siniestros = cartera.merge(
    siniestros[["poliza_id", "siniestro_id", "tipo_siniestro", "importe_pagado", "estado"]],
    how="left",
    on="poliza_id",
)

print(f"Filas de cartera: {len(cartera)}")
print(f"Filas tras el merge (left): {len(cartera_con_siniestros)}")
print(f"Pólizas sin ningún siniestro (importe_pagado nulo): {cartera_con_siniestros['importe_pagado'].isnull().sum()}")

Filas de cartera: 3000
Filas tras el merge (left): 3259
Pólizas sin ningún siniestro (importe_pagado nulo): 2059


El número de filas **crece** respecto a `cartera` porque una póliza con dos siniestros aparece dos veces (una por cada siniestro emparejado) — es el comportamiento esperado de un join uno-a-muchos, no un error.

### `how="inner"`: solo lo que coincide en ambas tablas

In [8]:
cartera_con_siniestros_inner = cartera.merge(
    siniestros[["poliza_id", "siniestro_id", "tipo_siniestro", "importe_pagado"]],
    how="inner",
    on="poliza_id",
)
print(f"Filas tras el merge (inner): {len(cartera_con_siniestros_inner)}")
print(f"¿Coincide con el total de siniestros? {len(cartera_con_siniestros_inner) == len(siniestros)}")

Filas tras el merge (inner): 1200
¿Coincide con el total de siniestros? True


Con `how="inner"`, el resultado tiene tantas filas como `siniestros` (cada siniestro encuentra su póliza, porque todos los `poliza_id` de `siniestros.csv` existen en `cartera_polizas.csv` — son datasets generados para ser coherentes entre sí). `how="right"` y `how="outer"` completan el resto de combinaciones posibles (equivalente a intercambiar la tabla "principal", o a quedarte con todo de ambas).

> 📊 **En Excel esto sería...** un `BUSCARV`/`XLOOKUP` arrastrado por toda la columna, pero trayendo varias columnas a la vez en vez de una por fórmula, y con la ventaja añadida de que `.merge()` deja explícito si el resultado debe conservar todas las filas de la tabla principal (`left`) o solo las que coinciden en ambas (`inner`) — en Excel, ese matiz depende de cómo gestionéis a mano los errores `#N/A` del `BUSCARV`.

**Para ti:** haced el mismo `merge` pero con `left_on`/`right_on` en vez de `on` (aunque aquí la columna se llame igual en ambos DataFrames, es la forma que necesitaréis cuando la clave tenga nombres distintos en cada tabla, como visteis en el Día 7 con `week1.merge(customers, left_on=["Customer ID"], right_on=["ID"])` del material original). Comprobad que el resultado es idéntico al `merge` con `on="poliza_id"`.

In [9]:
# Tu código aquí


## 3. `pivot_table()`: tablas dinámicas

[`pivot_table()`](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html) reorganiza un DataFrame "largo" (una fila por observación) en uno "ancho" (una fila por categoría de un campo, una columna por categoría de otro), agregando los valores con una función — literalmente una Tabla Dinámica de Excel.

Usamos `primas_mensuales.csv`: primas emitidas por año, mes y producto, en formato largo.

In [10]:
primas_mensuales.head()

,anio,mes,producto,num_polizas,primas_emitidas
0,2021,1,AUTO,52,22608.03
1,2021,2,AUTO,46,18287.48
2,2021,3,AUTO,52,22122.02
3,2021,4,AUTO,54,23579.10
4,2021,5,AUTO,58,24820.93


In [11]:
pivot_primas = primas_mensuales.pivot_table(
    values="primas_emitidas",
    index="mes",
    columns="anio",
    aggfunc="sum",
)
pivot_primas

anio,2021,2022,2023,2024
mes,,,,
1,45739.91,47591.83,52627.42,57694.36
2,40579.84,49077.87,50794.26,55737.30
3,47733.23,53330.50,57798.97,62587.35
4,47363.28,50969.70,57286.54,60340.60
5,49745.24,53770.73,60405.25,65957.61
6,53471.66,59313.35,63371.48,64850.53
7,54973.12,59502.21,60596.82,69807.60
8,40921.07,44025.98,46804.58,51016.21
9,50975.30,56651.46,61075.18,64991.50


Cada celda es la suma de `primas_emitidas` para esa combinación de mes (fila) y año (columna) — sumando los 4 productos (AUTO/HOGAR/VIDA/SALUD) de ese mes-año, porque no se pidió desglosarlos. Si quisiéramos ver el detalle por producto también, añadimos otra dimensión:

In [12]:
pivot_primas_producto = primas_mensuales.pivot_table(
    values="primas_emitidas",
    index=["anio", "mes"],
    columns="producto",
    aggfunc="sum",
).round(2)

pivot_primas_producto.head(8)

producto      AUTO     HOGAR     SALUD     VIDA
anio mes                                       
2021 1    22608.03   9583.72   8513.91  5034.25
     2    18287.48   8447.19   8719.43  5125.74
     3    22122.02   9536.66  10508.05  5566.50
     4    23579.10   9766.84   8925.12  5092.22
     5    24820.93   9634.41  10023.50  5266.40
     6    25937.04  10188.23  11358.78  5987.61
     7    24653.12  11346.65  11832.10  7141.25
     8    20292.00   7981.07   7730.73  4917.27

> 📊 **En Excel esto sería...** literalmente arrastrar `mes` al área de Filas, `anio` al área de Columnas y `primas_emitidas` al área de Valores (con "Suma" como función de resumen) en una Tabla Dinámica — `pivot_table()` es ese mismo flujo, en código, reproducible sin volver a hacer clic en nada si los datos cambian.

> ⚠️ **Error típico**: usar `aggfunc="mean"` (el valor por defecto de `pivot_table` si no se especifica) cuando en realidad hace falta `"sum"`, o viceversa. Con `primas_emitidas`, si hubiera más de una fila por combinación mes/año/producto (aquí no la hay, pero en datos reales es fácil que sí), la media y la suma dan números muy distintos y **ambos se ejecutan sin error** — solo uno es el que responde a la pregunta de negocio real ("¿cuánto se emitió en total?" pide `sum`, "¿cuál es el importe medio de una emisión?" pide `mean`).

## 4. Exportar a CSV y Excel

`.to_csv()` y `.to_excel()` son el reverso de `read_csv`/`read_excel` del Día 7. El parámetro más importante en ambos es `index` — por defecto `True`, lo que añade una columna extra con el índice del DataFrame que casi nunca queréis conservar al exportar.

In [13]:
pivot_primas_producto.reset_index().to_csv("_export_temporal.csv", index=False)

# Comprobamos que se ha guardado bien releyéndolo
pd.read_csv("_export_temporal.csv").head()

,anio,mes,AUTO,HOGAR,SALUD,VIDA
0,2021,1,22608.03,9583.72,8513.91,5034.25
1,2021,2,18287.48,8447.19,8719.43,5125.74
2,2021,3,22122.02,9536.66,10508.05,5566.50
3,2021,4,23579.10,9766.84,8925.12,5092.22
4,2021,5,24820.93,9634.41,10023.50,5266.40


In [14]:
import os
os.remove("_export_temporal.csv")  # limpiamos: es un archivo de ejemplo, no forma parte de data/raw/

> ⚠️ **Error típico**: exportar con `index=True` (el valor por defecto) sin darse cuenta, y luego, al releer el archivo, encontrarse con una columna `Unnamed: 0` que nadie pidió — es el índice numérico de pandas, guardado como si fuera una columna de datos más. Casi siempre se quiere `index=False` al exportar tablas que luego va a abrir alguien en Excel.

## 5. Con ayuda de la IA: validar una transformación `merge` + `groupby`

Vamos a pedirle a un asistente de IA una transformación real y algo más compleja: el **ratio de siniestralidad** por producto (importe total pagado en siniestros, dividido entre la prima anual total de ese producto). Es un indicador central en seguros, así que merece la pena que la validación sea sólida antes de confiar en el número.

### Prompt sugerido

In [15]:
prompt_ratio_siniestralidad = """
Contexto: tengo dos DataFrames de pandas. 'cartera' tiene una fila por
poliza, con columnas 'poliza_id', 'producto' y 'prima_anual'. 'siniestros'
tiene una fila por siniestro, con columnas 'poliza_id' (clave foranea a
cartera) e 'importe_pagado'. No todas las polizas tienen siniestros.

Objetivo: quiero calcular el ratio de siniestralidad por producto, definido
como (suma de importe_pagado de todos los siniestros de ese producto) /
(suma de prima_anual de todas las polizas de ese producto). El resultado
debe ser una Series (o DataFrame de una columna) indexada por producto.

Restricciones: el denominador debe usar la prima de TODAS las polizas de
ese producto (tengan o no siniestros), no solo las que aparecen en
'siniestros' tras un merge how='inner' -- si usas merge, tiene que ser
con cuidado de no perder polizas sin siniestros en el denominador.

Formato de salida: el codigo pandas (con groupby y/o merge), mas una frase
explicando por que el denominador no puede calcularse solo con un inner
join de ambas tablas.
"""
print(prompt_ratio_siniestralidad)


Contexto: tengo dos DataFrames de pandas. 'cartera' tiene una fila por
poliza, con columnas 'poliza_id', 'producto' y 'prima_anual'. 'siniestros'
tiene una fila por siniestro, con columnas 'poliza_id' (clave foranea a
cartera) e 'importe_pagado'. No todas las polizas tienen siniestros.

Objetivo: quiero calcular el ratio de siniestralidad por producto, definido
como (suma de importe_pagado de todos los siniestros de ese producto) /
(suma de prima_anual de todas las polizas de ese producto). El resultado
debe ser una Series (o DataFrame de una columna) indexada por producto.

Restricciones: el denominador debe usar la prima de TODAS las polizas de
ese producto (tengan o no siniestros), no solo las que aparecen en
'siniestros' tras un merge how='inner' -- si usas merge, tiene que ser
con cuidado de no perder polizas sin siniestros en el denominador.

Formato de salida: el codigo pandas (con groupby y/o merge), mas una frase
explicando por que el denominador no puede calcularse solo con 

**Qué se espera obtener**
- El numerador: siniestros agrupados por producto y sumados (`siniestros.groupby("producto")["importe_pagado"].sum()`).
- El denominador: prima agrupada por producto y sumada, calculado sobre **`cartera` directamente**, no sobre el resultado de un `merge`.
- Una explicación de por qué un `merge how="inner"` seguido de `groupby` daría un denominador **incorrecto**: un `inner join` solo conserva pólizas que sí tienen al menos un siniestro, así que la suma de `prima_anual` calculada después de ese join excluiría (sin avisar) todas las pólizas sin siniestros — infla artificialmente el ratio de siniestralidad, porque reduce el denominador sin razón.

**Qué debes verificar antes de darlo por bueno**
- Que el numerador y el denominador se calculan sobre bases de datos coherentes: el numerador debe ser la suma de siniestros de *ese* producto, el denominador la prima de *todas* las pólizas de *ese* producto — no un subconjunto filtrado sin querer por el propio `merge`.
- Que el resultado, para un producto conocido, coincide con un cálculo de control hecho a mano sobre un subconjunto pequeño.

In [16]:
# El cálculo "ingenuo" y potencialmente incorrecto: inner join, luego groupby sobre el resultado
merge_inner = cartera.merge(siniestros[["poliza_id", "importe_pagado"]], how="inner", on="poliza_id")

primas_tras_inner_join = merge_inner.groupby("producto")["prima_anual"].sum()
# OJO: esto sub-cuenta la prima real, porque solo incluye pólizas CON siniestro
print("Prima sumada tras el inner join (incorrecto como denominador):")
print(primas_tras_inner_join.round(2))

Prima sumada tras el inner join (incorrecto como denominador):
producto
AUTO     333903.88
HOGAR     72892.90
SALUD    135538.26
VIDA       4075.85
Name: prima_anual, dtype: float64


In [17]:
# El cálculo correcto: el denominador se calcula directamente sobre 'cartera', sin pasar por el merge
primas_totales_por_producto = cartera.groupby("producto")["prima_anual"].sum()
print("Prima total real por producto (correcto):")
print(primas_totales_por_producto.round(2))

# La diferencia demuestra el problema: el inner join siempre da un número MENOR o IGUAL
assert (primas_tras_inner_join <= primas_totales_por_producto[primas_tras_inner_join.index]).all()
print("\nConfirmado: la prima calculada tras el inner join es menor (excluye pólizas sin siniestro).")

Prima total real por producto (correcto):
producto
AUTO     598215.54
HOGAR    248151.05
SALUD    247504.70
VIDA     139853.19
Name: prima_anual, dtype: float64

Confirmado: la prima calculada tras el inner join es menor (excluye pólizas sin siniestro).


Con el denominador correcto, calculamos el ratio de siniestralidad y lo **validamos** contra un caso de control pequeño: el producto `VIDA`, que en este dataset tiene muy pocas pólizas y pocos siniestros, fácil de comprobar a mano.

In [18]:
siniestros_por_producto = siniestros.groupby("producto")["importe_pagado"].sum()

ratio_siniestralidad = (siniestros_por_producto / primas_totales_por_producto).round(4)
print("Ratio de siniestralidad por producto:")
print(ratio_siniestralidad)

Ratio de siniestralidad por producto:
producto
AUTO     2.0763
HOGAR    1.8333
SALUD    2.0853
VIDA     3.0573
dtype: float64


In [19]:
# Caso de control: recalculamos VIDA a mano, filtrando directamente (sin groupby, para tener otra vía de cálculo)
primas_vida_control = cartera.loc[cartera["producto"] == "VIDA", "prima_anual"].sum()
siniestros_vida_control = siniestros.loc[siniestros["producto"] == "VIDA", "importe_pagado"].sum()
ratio_vida_control = round(siniestros_vida_control / primas_vida_control, 4)

print(f"Prima total VIDA (control): {primas_vida_control:.2f}")
print(f"Siniestros totales VIDA (control): {siniestros_vida_control:.2f}")
print(f"Ratio VIDA (control): {ratio_vida_control}")

assert ratio_vida_control == ratio_siniestralidad["VIDA"],     "El ratio de siniestralidad de VIDA no coincide con el cálculo de control"
print("\nValidado: el ratio de siniestralidad coincide con el cálculo de control independiente.")

Prima total VIDA (control): 139853.19
Siniestros totales VIDA (control): 427569.09
Ratio VIDA (control): 3.0573

Validado: el ratio de siniestralidad coincide con el cálculo de control independiente.


El cálculo de control se hizo **sin** `groupby` ni `merge` (filtrando directamente por `producto == "VIDA"` y sumando) precisamente para que fuera una vía de cálculo distinta e independiente — si ambas vías dan el mismo número, es mucho más difícil que las dos compartan el mismo error por casualidad. Esta es la esencia de "validar contra un cálculo de control": no repetir la misma transformación con otras palabras, sino comprobarla desde un ángulo distinto.

## Resumen

- `pd.concat()` apila DataFrames por filas (o columnas, con `axis=1`); tras concatenar, revisad si el índice ha quedado duplicado (`ignore_index=True` o `keys=[...]` según lo que os interese conservar).
- `.merge()` une DataFrames por una clave común: `how="left"` conserva todas las filas de la tabla principal (como un `BUSCARV`/`XLOOKUP`, rellenando con `NaN` lo que no encuentra), `how="inner"` solo conserva lo que coincide en ambas.
- Un `merge` uno-a-muchos aumenta el número de filas del resultado respecto a la tabla "uno" — no es un error, es el comportamiento esperado.
- `pivot_table()` es la Tabla Dinámica de pandas: `index`/`columns`/`values`/`aggfunc` — presta atención a `aggfunc`, porque `"mean"` y `"sum"` dan números muy distintos y ambos se ejecutan sin error.
- `.to_csv()`/`.to_excel()` exportan resultados; usad `index=False` salvo que de verdad queráis conservar el índice como columna.
- Al calcular un ratio que combina dos tablas (como la siniestralidad), el denominador debe calcularse sobre la base completa, no sobre el resultado de un `merge how="inner"` que puede haber excluido registros sin coincidencia — y la validación más sólida es un cálculo de control hecho por una vía **distinta** a la transformación original, no una repetición de la misma.

**Siguiente:** `03_teoria_matplotlib_basico.ipynb`, donde entramos en la Unidad 8: visualización con Matplotlib.